# <center> Multi Index Mastery

**What is a MultiIndex?**\
**The Hierarchy Concept**\
Think:\
Company

├── HR\
│   ├── Male\
│   └── Female\
│\
└── IT\
    ├── Male\
    └── Female\
Instead of:\
One level\
we now have:\
Multiple levels


**Dataset**

In [2]:
import pandas as pd
df = pd.DataFrame({
    "Name":["Ahmed","Tooba","Mushtaq","Sania"],
    "Department":["HR","HR","IT","IT"],
    "Gender":["M","F","M","F"],
    "Salary":[50000,70000,80000,90000]
})
df

,Name,Department,Gender,Salary
0,Ahmed,HR,M,50000
1,Tooba,HR,F,70000
2,Mushtaq,IT,M,80000
3,Sania,IT,F,90000


**1. set_index()**\
This creates a MultiIndex

Single Index:

In [14]:
multi = df.set_index("Department")

Multiple Indexes:

In [16]:
multi = df.set_index(["Department","Gender"])

**Now:**\
Level 1 : Department\
Level 2 : Gender


**Understanding the Structure**\
**Internally:**\
(HR,M)\
(HR,F)\
(IT,M)\
(IT,F)\
These are tuples.\
The index becomes:\
MultiIndex([\
    ('HR','M'),\
    ('HR','F'),\
    ('IT','M'),\
    ('IT','F')\
])

**2. Accessing MultiIndexes with .loc**

Example: Get all HR employees

In [17]:
multi.loc["HR"]

,Name,Salary
Gender,,
M,Ahmed,50000
F,Tooba,70000


In [18]:
multi.loc[("HR","M")]

Name      Ahmed
Salary    50000
Name: (HR, M), dtype: object

**3. reset_index()**

In [19]:
multi

Name  Salary
Department Gender                 
HR         M         Ahmed   50000
           F         Tooba   70000
IT         M       Mushtaq   80000
           F         Sania   90000

Let's convert it back to flatenned rows

In [25]:
normal = multi.reset_index()

In [26]:
normal

,Department,Gender,Name,Salary
0,HR,M,Ahmed,50000
1,HR,F,Tooba,70000
2,IT,M,Mushtaq,80000
3,IT,F,Sania,90000


set_index() : columns --> indexes\
reset_index() : indexes --> columns

**4. GroupBy and MultiIndexes**\
Many times, GroupBy also creates multi indexes

In [31]:
result = df.groupby(
    ["Department","Gender"]
)["Salary"].mean()
result

Department  Gender
HR          F         70000.0
            M         50000.0
IT          F         90000.0
            M         80000.0
Name: Salary, dtype: float64

This multi index can also be reverted to normal form using reset_index()

In [32]:
result = result.reset_index()
result

,Department,Gender,Salary
0,HR,F,70000.0
1,HR,M,50000.0
2,IT,F,90000.0
3,IT,M,80000.0


There happens to be another professional method to avoid this MultiIndex issue:

In [36]:
pro_result = df.groupby(
    ["Department","Gender"],
    as_index=False
)["Salary"].mean()
pro_result

,Department,Gender,Salary
0,HR,F,70000.0
1,HR,M,50000.0
2,IT,F,90000.0
3,IT,M,80000.0


**5. stack()**

In [43]:
students = pd.DataFrame({
    "Math":[90,85],
    "Physics":[80,95]
},index = ["Ali","Sara"])
students

,Math,Physics
Ali,90,80
Sara,85,95


In [44]:
students = students.stack()

In [45]:
students

Ali   Math       90
      Physics    80
Sara  Math       85
      Physics    95
dtype: int64

Columns become another index level

**6. unstack()**\
The reverse operation

In [48]:
students = students.unstack()
students

,Math,Physics
Ali,90,80
Sara,85,95


So, somehow, stack() and unstack() work the same as create_index() and reset_index()

**7. MultiIndex Columns**\
Rows are not the only things that have multiple levels.\
Columns can too.

In [53]:
data = {
    ("Math","Midterm"):[90,85],
    ("Math","Final"):[95,88],
    ("Physics","Midterm"):[80,92],
    ("Physics","Final"):[85,96]
}
marks = pd.DataFrame(
    data,
    index=["Ali","Sara"]
)

In [54]:
marks

Math       Physics      
     Midterm Final Midterm Final
Ali       90    95      80    85
Sara      85    88      92    96

Getting all maths scores

In [55]:
marks["Math"]

,Midterm,Final
Ali,90,95
Sara,85,88


Getting all physics scores

In [56]:
marks["Physics"]

,Midterm,Final
Ali,80,85
Sara,92,96


Getting only Maths Final

In [58]:
marks[("Math","Final")]

Ali     95
Sara    88
Name: (Math, Final), dtype: int64

**8. swaplevel()**

In [59]:
multi

Name  Salary
Department Gender                 
HR         M         Ahmed   50000
           F         Tooba   70000
IT         M       Mushtaq   80000
           F         Sania   90000

In [63]:
multi.swaplevel()

,,Name,Salary
Gender,Department,,
M,HR,Ahmed,50000
F,HR,Tooba,70000
M,IT,Mushtaq,80000
F,IT,Sania,90000


**9. sort_index()**

In [66]:
multi.swaplevel().sort_index()

Name  Salary
Gender Department                 
F      HR            Tooba   70000
       IT            Sania   90000
M      HR            Ahmed   50000
       IT          Mushtaq   80000

**10. xs() (Cross Sections)**\
This is an advanced but very useful operation

In [67]:
multi

Name  Salary
Department Gender                 
HR         M         Ahmed   50000
           F         Tooba   70000
IT         M       Mushtaq   80000
           F         Sania   90000

Cut through Gender and give me only Males

In [68]:
multi.xs(
    "M",
    level="Gender"
)

,Name,Salary
Department,,
HR,Ahmed,50000
IT,Mushtaq,80000


Cut through Department and give me only HR

In [69]:
multi.xs(
    "HR",
    level="Department"
)

,Name,Salary
Gender,,
M,Ahmed,50000
F,Tooba,70000


xs means cross section which means cut through